Install Required Libraries

In [ ]:
pip install pandas mysql-connector-python matplotlib openai

Connect to MySQL Database

In [ ]:
import pandas as pd
import mysql.connector
import matplotlib.pyplot as plt
from openai import OpenAI

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="hr_dashboard"
)

query = """
SELECT 
    e.employee_id,
    e.department,
    e.joining_date,
    r.resignation_date,
    r.reason
FROM employee e
LEFT JOIN resignation r
ON e.employee_id = r.employee_id
"""

df = pd.read_sql(query, conn)

df.head()

Calculate Attrition Rate

Total Employees per Department

In [ ]:
total_employees = df.groupby("department")["employee_id"].count()

Employees Who Resigned

In [ ]:
resigned = df[df["resignation_date"].notnull()]
resigned_by_dept = resigned.groupby("department")["employee_id"].count()

Attrition Rate Calculation

In [ ]:
attrition_rate = (resigned_by_dept / total_employees) * 100
attrition_rate = attrition_rate.fillna(0)
print(attrition_rate)

Analyze Reason for Leaving

In [ ]:
reason_counts = resigned["reason"].value_counts()
print(reason_counts)

Attrition Rate by Department

In [ ]:
attrition_rate.plot(kind="bar")

plt.title("Attrition Rate by Department")
plt.xlabel("Department")
plt.ylabel("Attrition Rate (%)")

plt.show()

Reasons for Leaving

In [ ]:
reason_counts.plot(kind="bar")

plt.title("Reasons for Employee Resignation")
plt.xlabel("Reason")
plt.ylabel("Number of Employees")

plt.show()

LLM Insights with OpenAI

In [ ]:
client = OpenAI()

data_summary = attrition_rate.to_string()

response = client.responses.create(
    model="gpt-4.1-mini",
    input=f"""
Analyze employee attrition data by department and suggest HR improvements:

{data_summary}
"""
)

print(response.output_text)